In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zhMin100.H", "h"))
    m = np.array([open("../tmp/news2016zhMin100.txt").read()]).view("U1")

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)

    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class Transpose(Module):
    @staticmethod
    def forward(x):
        return x.transpose(1, 2)


class ConvBn(Sequential):
    def __init__(s, i, w=3):
        super().__init__(Conv1d(i, i, w, padding=w // 2, bias=0), BatchNorm1d(i, track_running_stats=0), ReLU(1))


class ConvTransBn(Sequential):
    def __init__(s, i, w=3):
        super().__init__(ConvTranspose1d(i, i, w, padding=w // 2, bias=0), BatchNorm1d(i, track_running_stats=0), ReLU(1))


class Res(Sequential):
    def forward(s, x):
        return super().forward(x) + x


class Cnn(Sequential):
    def __init__(s, *a, n=0):
        super().__init__(*a)
        s.n = n

    def forward(s, x):
        for _ in range(s.n or int(math.log2(x.size(2)))):
            x = super().forward(x)
        return x


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, y):
        return super().forward(x.view(-1, x.size(-1)), y.ravel())

In [ ]:
m = Learner(
    DataLoaders(Dat(16, 2**13*3, 10), Dat(16, 2**13*3, 1000)),
    Sequential(
        Embedding(684, 32),
        Transpose(),
        Cnn(Res(ConvBn(32), ConvBn(32)), MaxPool1d(2)),
        ConvBn(32, 1),
        Conv1d(32, 32, 1, bias=0),
        Cnn(Res(ConvTransBn(32), ConvTransBn(32)), ConvTranspose1d(32, 32, 3, 2, 1, 1), n=4),
        ConvBn(32, 1),
        Transpose(),
        Linear(32, 684),
    ).cuda(),
    MultiCrossEntropyLoss(),
    cbs=[
        ShowGraphCallback,
        SaveModelCallback(fname="model5f", with_opt=1, reset_on_fit=0),
    ],
    metrics=accuracy,
    wd=0.01,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(30, .036, 9, 99, .01)

In [ ]:
def tree(a):
    a = list(a)
    d = int(math.log2(len(a) + 1))
    print(len(a))
    for i in range(d):
        for j in range(2 ** i):
            if j:
                print(end="  " * (2 ** (d - i - 2) - 1))
            print(end=a.pop(0))
        print()


x, y = next(iter(m.dls[0]))
x = m.model(x)[0]
y = y[0]
print(m.dls.m[y.cpu()].view("U8")[0])
tree(m.dls.m[x[:, :684].argmax(1).cpu()])